In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# os.environ["JAX_PLATFORM_NAME"] = "cpu"

In [3]:
import h5py, os, tqdm, glob, scipy
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, Dopri5, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize
from jax.experimental.ode import odeint

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.pm import linear_field, lpt, make_ode_fn, pm_forces, make_ode_fn_diffrax, make_ode_fn
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, ResNet3D, ResNetBlock3D, GraphConvolution, CNN, HybridNet, AttentionGNN
from jaxpm import camels, plotting, hpm, nn, graph

# print(jax.devices("gpu"))
print(jax.default_backend())

gpu


# configuration

In [4]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# CAMELS

In [5]:
CAMELS = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims"
# CODE = "IllustrisTNG"
CODE = "SIMBA"
# CODE = "Astrid"
SIMSET = "CV"
CV = "CV_0"

out_dict = camels.load_CV_snapshots(
    os.path.join(CAMELS, CODE, SIMSET, CV),
    mesh_per_dim,
    parts_per_dim,
    # i_snapshots=[-2,-1],
    i_snapshots=range(1, 33+4, 8),
    # i_snapshots=range(1, 33+4, 4),
    return_hydro=True,
)

cosmo = out_dict["cosmo"]
scales = out_dict["scales"]

dm_poss = out_dict["dm_poss"]
dm_vels = out_dict["dm_vels"]

gas_poss = out_dict["gas_poss"]
gas_vels = out_dict["gas_vels"]
gas_masss = out_dict["gas_masss"]

Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_018.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_042.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_058.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_074.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


finding unique gas particle indices:  20%|██        | 1/5 [00:01<00:05,  1.36s/it]

Found 368 duplicate gas particle IDs


finding unique gas particle indices:  40%|████      | 2/5 [00:07<00:11,  3.93s/it]

Found 4088 duplicate gas particle IDs


...

finding unique gas particle indices:  80%|████████  | 4/5 [00:18<00:05,  5.04s/it]

Found 14787 duplicate gas particle IDs


finding unique gas particle indices: 100%|██████████| 5/5 [00:23<00:00,  4.76s/it]


There are 15915410 (94.86%) gas particles that exist in all snapshots


loading snapshots: 100%|██████████| 5/5 [00:56<00:00, 11.23s/it]


In [6]:
@nnx.jit(static_argnames=("loss_fn",))
def train_step(model, optimizer, loss_fn):
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss

losses = []

In [9]:
# latent_init = cic_read(cic_paint(jnp.zeros(mesh_shape), gas_poss[0]), gas_poss[0])
# latent_init = jnp.expand_dims(latent_init, axis=-1)

# latent_init = jnp.ones((parts_per_dim**3,1))
latent_init = jnp.ones((parts_per_dim**3,16))

# latent_init = None

# mass_init = gas_masss[0] - cosmo.Omega_b / (cosmo.Omega_c + cosmo.Omega_b)

edges = graph.get_edges(gas_poss, scales, k=4)

def solve_ode_diffrax(model, architecture):
    res = diffeqsolve(
            terms=ODETerm(hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, gas_model=model, gas_architecture=architecture, precomputed_edges=edges)),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=0.02,
            # y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
            y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0], latent_init),
            # y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0], mass_init),
            saveat=SaveAt(ts=scales),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )

    res = res.ys

    return res

In [10]:
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

# loss

### CAMELS ground truth

In [11]:
gas_mass = cosmo.Omega_b / (cosmo.Omega_c + cosmo.Omega_b)
dm_mass = cosmo.Omega_c / (cosmo.Omega_c + cosmo.Omega_b)

# per-particle reference
ref_pos = jnp.stack(gas_poss, axis=0)
ref_vel = jnp.stack(gas_vels, axis=0)

# field-level reference
ref_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

ref_mass = gas_masss

# power spectrum reference
vpower_spectrum = jax.vmap(
    lambda fields: 
        power_spectrum(
            compensate_cic(fields),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
_, ref_cls = vpower_spectrum(ref_rho)

vcross_correlation_separate = jax.vmap(
    lambda field_a, field_b:
        cross_correlation_coefficients(
            compensate_cic(field_a),
            compensate_cic(field_b),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
vcross_correlation = lambda rhos: vcross_correlation_separate(rhos, ref_rho)

### particle-level

In [12]:
def particle_loss_fn(model, architecture):
    res = solve_ode_diffrax(model, architecture)
    gas_poss = res[2]
    gas_vels = res[3]

    delta_pos = ((gas_poss - ref_pos + mesh_per_dim // 2) % mesh_per_dim) - mesh_per_dim // 2
    pos_loss = jnp.sum(delta_pos**2, axis=-1)
    pos_loss = jnp.mean(pos_loss)

    vel_disp = jnp.std(gas_vels, axis=0)
    vel_loss = jnp.sum(((gas_vels - ref_vel) / (vel_disp + 1e-5))**2, axis=-1)
    vel_loss = jnp.mean(vel_loss)

    res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
    _, res_cls = vpower_spectrum(res_rho)
    cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))

    _, res_cross = vcross_correlation(res_rho)
    cross_loss = jnp.mean(jnp.sum((res_cross/jnp.sqrt(ref_cls * res_cls) - 1)**2, axis=-1))

    # mass_loss = jnp.mean((res[4] + cosmo.Omega_b / (cosmo.Omega_c + cosmo.Omega_b) - ref_mass)**2)

    # return pos_loss + 0.1 * cl_loss + mass_loss
    return pos_loss + 0.1 * cl_loss
    # return pos_loss + 0.01 * vel_loss + 0.01 * cl_loss
    # return pos_loss + 0.01 * vel_loss + 0.1 * cl_loss
    # return pos_loss + 0.01 * vel_loss + 0.1 * cl_loss + 1 * cross_loss
    # return cl_loss
    # return cross_loss
    # return cl_loss
    # return pos_loss + 0.01 * vel_loss
    # return pos_loss + 0.01 * vel_loss
    # return pos_loss + vel_loss
    # return pos_loss


### field-level

In [13]:
def field_loss_fn(model, architecture):
    res = solve_ode_diffrax(model, architecture)

    rho = vcic_paint(jnp.zeros(mesh_shape), res[2], cosmo.Omega_b / cosmo.Omega_c)
    
    # rho_loss = jnp.nanmean((rho - ref_rho)**2)
    
    eps = 1e-5
    rho_loss = jnp.mean((jnp.log1p(rho + eps) - jnp.log1p(ref_rho + eps))**2)
    
    # _, res_cls = vpower_spectrum(res_rho)
    # cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))
    
    return rho_loss
    # return rho_loss + 0.1 * cl_loss


# architecture

### MLP

In [14]:
# model = MLP(
#     d_in=5 + latent_init.shape[-1],
#     d_out=1 + latent_init.shape[-1], 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )
# architecture = "mlp"

In [15]:
# with_latent = False

# model = MLP(
#     d_in=5 + with_latent, 
#     # d_in=3 + with_latent, 
#     d_out=1 + with_latent, 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )
# architecture = "mlp"

### MLP + CNN

In [16]:
# with_latent = False

# mlp = MLP(
#     d_in=5 + with_latent,
#     d_out=8, 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )

# cnn = CNN(
#     d_in=4 + with_latent,
#     d_out=8,
#     d_hidden=8,
#     n_hidden=2,
#     kernel_size=(3, 3, 3),
#     strides=1,
#     rngs=nnx.Rngs(0)
# )

# model = HybridNet(
#     mlp,
#     cnn,
#     d_out=1 + with_latent,
#     rngs=nnx.Rngs(0)
# )

# architecture = "mlp+cnn"

### GNN

on the fly

In [17]:
with_latent = False

model = AttentionGNN(
    d_node=5 + with_latent,
    d_edge=1,
    d_query=16,
    n_hidden=4,
    d_out=1 + with_latent,
    rngs=nnx.Rngs(0),
)

architecture = "gnn"

# training

In [18]:
total_steps = 300
learning_rate = 1e-4
# learning_rate = optax.cosine_decay_schedule(
#     init_value=1e-3, 
#     decay_steps=total_steps, 
#     alpha=0.1
# )
clip_norm = 1

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
    )
)

losses = []
loss_fn = lambda model: particle_loss_fn(model, architecture)
# loss_fn = lambda model: field_loss_fn(model, architecture)

In [19]:
for i in (pbar := tqdm.tqdm(range(total_steps))):  
    loss = train_step(model, optimizer, loss_fn)

    losses.append(loss)
    pbar.set_description(f"Loss: {loss:.4f}")

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log")

  0%|          | 0/300 [00:00<?, ?it/s]


dark matter, gas and latent
Using learned pressure force


TypeError: dot_general requires contracting dimensions to have the same shape, got (5,) and (21,).

### checkpointing

In [ ]:
# # see https://flax.readthedocs.io/en/latest/guides/checkpointing.html
# # checkpoint_file = os.path.join(os.getcwd(), "checkpoints/hpm_mlp_sim.jx")
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/hpm_mlp_sim_new_2.jx")
# checkpointer = ocp.StandardCheckpointer()
# print(os.getcwd())

In [ ]:
# _, params = nnx.split(model)
# checkpointer.save(checkpoint_file, params, force=True)

In [ ]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

# run the simulation

In [20]:
og_ode = hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo)

og_res = diffeqsolve(
        terms=ODETerm(og_ode),
        solver=LeapfrogMidpoint(),
        t0=scales[0],
        t1=scales[-1],
        dt0=0.01,
        y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
        saveat=SaveAt(ts=scales),
        max_steps=100,
        stepsize_controller=ConstantStepSize(),
)

og_dm_poss, og_dm_vels, og_gas_poss, og_gas_vels = og_res.ys

dark matter and gas


UnboundLocalError: cannot access local variable 'd_gas_latent' where it is not associated with a value

In [ ]:
nn_ode = hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, gas_model=model)

nn_res = diffeqsolve(
        terms=ODETerm(nn_ode),
        solver=LeapfrogMidpoint(),
        t0=scales[0],
        t1=scales[-1],
        dt0=0.01,
        # y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
        y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0], mass_init),
        saveat=SaveAt(ts=scales),
        max_steps=100,
        stepsize_controller=ConstantStepSize(),
)

# nn_dm_poss, nn_dm_vels, nn_gas_poss, nn_gas_vels = nn_res.ys
nn_dm_poss, nn_dm_vels, nn_gas_poss, nn_gas_vels, nn_gas_mass = nn_res.ys

In [ ]:
# fig, ax = plt.subplots()

# j = np.random.choice(np.arange(nn_gas_mass.shape[1]), 20)
# ax.plot(scales, nn_gas_mass[:,j])
# ax.set(xlabel="scale", ylabel="gas mass")

In [ ]:
plotting.compare_particle_evolution(
    mesh_shape, 
    scales, 
    jnp.stack([gas_poss, og_gas_poss, nn_gas_poss], axis=0), 
    title="gas",
    col_titles=["CAMELS", "gravity", "gravity + pressure"],
    include_pk=True,
    include_reference=True,
)